In [43]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'outputs/' 가 만들어집니다)
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.0.1
pandas: 2.3.1
저장 폴더: /Users/simoncho/Desktop/DI-2/D009/outputs


In [44]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터

# 난수 시드 고정: 실행할 때마다 같은 무작위 데이터가 나오게 함 (결과 재현용)
np.random.seed(13)

n_days = 90                        # 기간: 90일
n_stores = 6                       # 매장 수: 6개
n_rows = n_days * n_stores * 4     # 매장 x 일자 x 시간대(4구간) = 2,160행

# 매장 ID 목록: {i:02d} → 숫자를 두 자리로 맞춤 → 'S01', 'S02', ..., 'S06'
stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]

# 판매 상품 목록 (정상 표기 5종)
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

# 빵셀러 판매 데이터 DataFrame 생성
bakery = pd.DataFrame({
    # np.tile: 매장 목록을 반복해 이어 붙임 → S01~S06이 순서대로 반복
    # [:n_rows]: 길이를 정확히 n_rows로 맞춤
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],

    # 날짜 문자열: 날짜 3개를 형식 3가지(하이픈 / 슬래시 / 구분자 없음)로 섞어 무작위 배정
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),

    # 상품: 5종 중 무작위 배정
    "item": np.random.choice(items, n_rows),

    # 수량: 1, 2를 두 번씩 넣어 더 자주 나오게 함
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값

    # 단가: 5가지 가격 중 무작위 배정
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})

# 매출 = 수량 × 단가
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
# ① 결측: 무작위 60개 행의 매출을 NaN으로 (replace=False → 같은 행 중복 선택 방지)
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan

# ② 음수: 인덱스 5번 행의 매출을 -45,000으로
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수

# ③ 공백: 무작위 10개 행의 매장 ID를 앞뒤 공백이 붙은 ' S01 '로 (random_state=1 → 같은 행이 뽑히도록 고정)
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백

# ④ 표기 혼재: 무작위 8개 행의 상품명을 '케익'으로 (random_state=2 → 같은 행이 뽑히도록 고정)
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')

# ⑤ 중복: 0~3번 행을 복사해 맨 뒤에 붙임 (ignore_index=True → 인덱스를 0부터 새로 매김)
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

# 결과 확인: 2,160행 + 중복 4행 = (2164, 6)
print("빵셀러 데이터 생성 완료:", bakery.shape)

# 앞 5행 미리보기
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


In [45]:
# CSV로 임시 저장(인제스트 단계에서 읽어옴) — 분석가가 "파일을 받았다" 상황 재현

# 원본 파일을 둘 폴더 경로 만들기: OUT_DIR 아래 raw 폴더
#RAW_DIR = OUT_DIR / "outputs"

# 폴더 생성
#   parents=True  → OUT_DIR가 아직 없어도 상위 폴더까지 한 번에 만들어 줌
#   exist_ok=True → 이미 폴더가 있어도 에러 내지 않고 넘어감
#RAW_DIR.mkdir(parents=True, exist_ok=True)

# 저장 대상은 리스트(stores, items)가 아니라 DataFrame인 bakery
#   index=False         → 0, 1, 2... 행 번호는 저장하지 않음 (다시 읽을 때 쓸데없는 컬럼이 생기는 것 방지)
#   encoding="utf-8-sig" → 한글('크루아상', '케익')이 엑셀에서 열어도 깨지지 않게 함
bakery.to_csv(OUT_DIR / "bakery.csv", index=False, encoding="utf-8-sig")

# 저장 결과 확인용 출력
print("종합 프로젝트 데이터 생성 및 CSV 저장 완료")
print(f"  bakery: {bakery.shape}")             # (행 수, 열 수) — 중복 4건 포함해 2164행이어야 함

# .resolve() → 상대 경로를 전체 경로로 바꿔서, 실제로 어느 폴더에 저장됐는지 보여줌
print(f"\n저장 경로: {OUT_DIR.resolve()}")



종합 프로젝트 데이터 생성 및 CSV 저장 완료
  bakery: (2164, 6)

저장 경로: /Users/simoncho/Desktop/DI-2/D009/outputs


In [46]:
# 품질 리포트 함수 — 컬럼별 결측·고유값·이상치·음수·공백·날짜 의심 여부를 한 표로 진단
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.

    Parameters
    ----------
    df   : 진단할 DataFrame
    name : 출력 메시지에 표시할 데이터 이름

    Returns
    -------
    컬럼별 진단 결과 DataFrame (행 = 원본의 컬럼)
    '''
    n_rows = len(df)  # 전체 행 수 (출력용)

    # ── 1. 기본 진단: 컬럼마다 타입, 결측 수, 결측 비율, 고유값 수 ──
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),                     # 컬럼 타입 (날짜가 object면 변환 필요 신호)
        "missing": df.isna().sum(),                         # 결측 개수
        "missing_pct": (df.isna().mean() * 100).round(2),   # 결측 비율(%) — isna()의 평균 = 결측 비율
        "n_unique": df.nunique(dropna=True),                # 고유값 수 — 예상보다 많으면 표기 오염 의심
    })

    # ── 2. 수치형 컬럼: IQR 이상치 비율 + 음수 개수 ──
    outlier_pct = {}
    negative = {}                                           # [추가] 음수 개수 (환불·입력 실수 탐지)
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()                                # 결측은 빼고 계산
        q1, q3 = s.quantile(0.25), s.quantile(0.75)         # 1사분위, 3사분위
        iqr = q3 - q1                                       # 사분위 범위 (가운데 50%의 폭)
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr             # 이상치 경계 (Tukey 기준)
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100   # 경계 밖 비율(%)
        negative[col] = (s < 0).sum()                       # [추가] 0보다 작은 값 개수
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)   # 문자열 컬럼은 자동으로 NaN
    base["negative"] = pd.Series(negative)                       # [추가]

    # ── 3. 문자열 컬럼: 앞뒤 공백 + 날짜 의심 여부 ──
    obj_cols = df.select_dtypes(include=["object", "string"]).columns   # pandas 3에서도 동작
    whitespace = {}                                         # [추가] 앞뒤 공백이 있는 값 개수
    suspicious_datetime = []
    for col in obj_cols:
        s = df[col].dropna().astype(str)
        whitespace[col] = (s != s.str.strip()).sum()        # [추가] strip 전후가 다르면 공백이 있는 값

        try:
            # 진단 단계라 format="mixed"로 여러 날짜 형식을 넓게 허용
            # (실제 정제할 땐 형식을 확인해서 format을 정확히 지정)
            parsed = pd.to_datetime(df[col], errors="coerce", format="mixed")
            if parsed.notna().mean() > 0.8:                 # 80% 이상 날짜로 변환되면 날짜 컬럼으로 의심
                suspicious_datetime.append(col)
        except Exception as e:
            print(f"  (날짜 판별 실패: {col} — {e})")       # 조용히 넘기지 않고 이유를 남김
    base["whitespace"] = pd.Series(whitespace)                       # [추가]
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)    # 날짜 의심 컬럼이면 True

    # ── 4. 요약 출력 ──
    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")    # 모든 컬럼 값이 같은 행 (우연한 중복도 포함됨)
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")

    return base  # 진단표를 돌려줘서 저장하거나 후속 처리에 재사용


# 저장해 둔 CSV를 "받은 파일"처럼 다시 읽어서 진단
bakery_csv = pd.read_csv(OUT_DIR / "bakery.csv")
qr_bakery_full = quality_report_full(bakery_csv, "bakery_csv")
qr_bakery_full

[품질 리포트(완전판)] bakery_csv
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건
  📌 날짜로 보이는 object 컬럼: ['date_str']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,negative,whitespace,maybe_datetime
store_id,object,0,0.00,7,NaN,NaN,10.0,False
date_str,object,0,0.00,9,NaN,NaN,0.0,True
item,object,0,0.00,6,NaN,NaN,0.0,False
quantity,int64,0,0.00,5,14.56,0.0,NaN,False
unit_price,int64,0,0.00,5,0.00,0.0,NaN,False
revenue,float64,60,2.77,26,17.30,1.0,NaN,False


In [47]:
# 정제 파이프라인
# ── 단계별 정제 함수 정의 (각 함수는 DataFrame을 받아 정제된 DataFrame을 돌려줌) ──

# 1. 완전 중복 제거
#    drop_duplicates(): 모든 컬럼 값이 똑같은 행 중 첫 번째만 남기고 삭제
#    reset_index(drop=True): 삭제로 듬성듬성해진 인덱스를 0부터 다시 매김 (기존 인덱스는 컬럼으로 남기지 않음)
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True)

# 2. store_id 앞뒤 공백 제거
#    .str.strip(): 문자열 앞뒤 공백 제거 → ' S01 ' → 'S01'
#    assign(): 원본을 바꾸지 않고, 해당 컬럼을 바꾼 새 DataFrame을 돌려줌
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip())

# 3. item 표기 통일
#    replace({바꿀 값: 새 값}): 딕셔너리에 적힌 값만 바꾸고 나머지는 그대로 둠 → '케익' → '케이크'
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"}))

# 4. 날짜 파싱
#    pd.to_datetime(): 문자열 → datetime 변환
#      format="mixed"   → '2025-04-01', '2025/04/01', '20250401'처럼 섞인 형식을 행마다 알아서 해석
#      errors="coerce"  → 변환 실패한 값은 에러 대신 NaT(날짜 결측)로
#    새 컬럼 date를 만든 뒤, 더 이상 필요 없는 원본 date_str 컬럼은 삭제
def b_parse_date(df): return df.assign(
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"])


# 5-1. 환불(음수 매출) 행을 미리 따로 보관
#      파이프라인에서 지우기 전에 원본에서 골라 둬야 나중에 환불 분석에 쓸 수 있음
#      .copy(): 원본과 연결이 끊긴 독립 복사본 → 이후 수정해도 원본에 영향 없고 경고도 안 뜸
refunds_bakery = bakery[bakery["revenue"] < 0].copy()


# ── 파이프라인: 위 함수들을 .pipe()로 순서대로 적용 ──
bakery_clean = (
    bakery
    .pipe(b_dedup)          # 1. 완전 중복 제거
    .pipe(b_strip_store)    # 2. 매장 ID 공백 제거
    .pipe(b_unify_item)     # 3. 상품명 통일
    .pipe(b_parse_date)     # 4. 날짜 문자열 → datetime

    # 5-2. 본 분석용 데이터에서는 음수 매출 제외
    #      lambda d: ... → 이름 없는 한 줄 함수, d는 앞 단계까지 정제된 DataFrame
    #      (참고: NaN >= 0은 False라서 이 단계에서 결측 행도 함께 빠짐)
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외

    # 6. 매출 결측 행 제거
    #    subset=["revenue"] → revenue 컬럼에 결측이 있는 행만 삭제 (다른 컬럼 결측은 상관없음)
    .pipe(lambda d: d.dropna(subset=["revenue"]))

    # 필터링으로 비어 버린 인덱스를 0부터 다시 매김
    .reset_index(drop=True)
)


# ── 결과 확인 ──
print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")   # (행, 열) 변화
print(f"환불 보관: {len(refunds_bakery)}건")                           # 따로 보관한 음수 매출 건수
print(f"item 종류: {bakery_clean['item'].unique()}")                  # '케익'이 사라지고 5종만 남았는지 확인

정제 전: (2164, 6)  →  정제 후: (1784, 6)
환불 보관: 1건
item 종류: ['쿠키' '케이크' '샌드위치' '크루아상' '식빵']


In [49]:
# KPI 표로 변환한 뒤, 각각 Parquet으로 저장
#   월별·매장별 매출 합계 (행=매장, 열=월)
#   상품별 평균 단가·총매출

# ── KPI 1: 월별·매장별 매출 합계 (wide 형태: 행=매장, 열=월) ──

# 날짜에서 '월' 컬럼 만들기
#   .dt.to_period("M") → 날짜를 월 단위 기간으로 변환 (2025-04-01 → 2025-04)
#   .astype(str)       → '2025-04' 같은 문자열로 바꿈 (표의 열 이름으로 쓰기 좋고 Parquet 저장도 안전)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)

store_month = (
    bakery_clean
    # 매장 + 월 조합으로 묶어서 매출 합계 → 결과는 (store_id, month) 2단 인덱스를 가진 Series (long 형태)
    .groupby(["store_id", "month"])["revenue"].sum()
    # unstack(): 인덱스의 마지막 단계(month)를 열로 올림 → 행=매장, 열=월인 표(wide 형태)로 바뀜
    #   fill_value=0 → 특정 매장에 어떤 달 매출 기록이 없으면 NaN 대신 0으로 채움
    .unstack(fill_value=0)
    # 소수점 없이 원 단위로 반올림
    .round(0)
)

print("월별·매장별 매출:")
display(store_month)   # 주피터에서 표 형태로 출력 (셀 중간에서도 표가 보이게 함)


# ── KPI 2: 상품별 평균 단가·총매출 ──
item_kpi = (
    bakery_clean
    .groupby("item")                                   # 상품별로 묶기
    .agg(                                              # named aggregation: 결과컬럼명=(원본컬럼, 함수)
        avg_price=("unit_price", "mean"),              # 평균 단가
        total_revenue=("revenue", "sum"),              # 총매출
        n_orders=("revenue", "count"),                 # 판매 건수 (count는 결측 제외 개수 = 행 수)
    )
    .round(0)                                          # 원 단위로 반올림
    .sort_values("total_revenue", ascending=False)     # 총매출이 큰 상품부터 정렬
)

print("\n상품별 KPI:")   # \n → 앞에 빈 줄 하나 넣어 출력 구분
display(item_kpi)


# ── Parquet 저장 ──
# index=False를 쓰지 않은 이유: 두 표 모두 인덱스(store_id, item)가 핵심 정보라서 함께 저장해야 함
# (Parquet은 인덱스와 dtype을 보존 → 다시 읽으면 같은 표 모양 그대로 복원됨)
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")

# .resolve() → 실제 저장된 폴더의 전체 경로를 보여줌
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6834000.0,7243500.0,5884000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S04,6471500.0,7105000.0,6744500.0
S05,8681500.0,4102000.0,4960000.0
S06,9156000.0,5095500.0,6796500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: /Users/simoncho/Desktop/DI-2/D009/outputs


In [53]:
# 결정 로그 표
decisions = []


def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})


log_decision(
    "1. 중복 제거",
    "완전 중복 행 drop",
    "동일 키 동일 값은 시스템 입력 오류로 간주",
    "321건 제거",
)
log_decision(
    "2. 문자열 정제",
    "store_id를 strip",
    "'케익'은 '케이크'과 같은 item",
    "item 종류 6→5로 통일",
)
log_decision(
    "3. 날짜 파싱",
    "head로 확인결과 date_str을 datetime으로 판단, date_str를 datetime으로",
    "월별 집계가 분석 목적이므로 datetime 필요",
    f"NaT(파싱 실패): {bakery_clean['date'].isna().sum()}건",
)
log_decision(
    "4. 이상치 처리",
    "음수 amount 분리, 수량 상위 1% 컷오프",
    "매출 분석이 목적이므로 환불은 별도 보관",
    f"환불 {len(refunds_bakery)}건",
)
log_decision(
    "5. 결측치 처리",
    "revenue 결측 행 제거",
    "결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",
    "최종 60건 내외 제거",
)

decision_log = pd.DataFrame(decisions)
decision_log

,step,action,reason,result
0,1. 중복 제거,완전 중복 행 drop,동일 키 동일 값은 시스템 입력 오류로 간주,321건 제거
1,2. 문자열 정제,store_id를 strip,'케익'은 '케이크'과 같은 item,item 종류 6→5로 통일
2,3. 날짜 파싱,"head로 확인결과 date_str을 datetime으로 판단, date_str를 ...",월별 집계가 분석 목적이므로 datetime 필요,NaT(파싱 실패): 0건
3,4. 이상치 처리,"음수 amount 분리, 수량 상위 1% 컷오프",매출 분석이 목적이므로 환불은 별도 보관,환불 1건
4,5. 결측치 처리,revenue 결측 행 제거,"결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",최종 60건 내외 제거


In [54]:
# 결정로그 마크다운 변환
md_table = "| 단계 | 액션 | 근거 | 결과 |\n|---|---|---|---|\n"
for d in decisions:
    md_table += f"| {d['step']} | {d['action']} | {d['reason']} | {d['result']} |\n"

print(md_table)

| 단계 | 액션 | 근거 | 결과 |
|---|---|---|---|
| 1. 중복 제거 | 완전 중복 행 drop | 동일 키 동일 값은 시스템 입력 오류로 간주 | 321건 제거 |
| 2. 문자열 정제 | store_id를 strip | '케익'은 '케이크'과 같은 item | item 종류 6→5로 통일 |
| 3. 날짜 파싱 | head로 확인결과 date_str을 datetime으로 판단, date_str를 datetime으로 | 월별 집계가 분석 목적이므로 datetime 필요 | NaT(파싱 실패): 0건 |
| 4. 이상치 처리 | 음수 amount 분리, 수량 상위 1% 컷오프 | 매출 분석이 목적이므로 환불은 별도 보관 | 환불 1건 |
| 5. 결측치 처리 | revenue 결측 행 제거 | 결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지 | 최종 60건 내외 제거 |



In [51]:
# 코드 퀴즈
# bakery_clean에서 item이 '케이크'이고 revenue >= 10000인 행만 골라서 store_id별 평균 revenue를 구한 뒤, Parquet으로 저장하세요.

cake_kpi = (
    bakery_clean
    # query(): 조건을 문자열로 써서 행을 필터링
    #   item == '케이크'  → 케이크 판매 행만 (문자열 값은 작은따옴표로 감쌈)
    #   and revenue >= 10000 → 그중 매출이 1만 원 이상인 건만
    #   (bakery_clean[(bakery_clean["item"] == "케이크") & (bakery_clean["revenue"] >= 10000)]과 같은 의미)
    .query("item == '케이크' and revenue >= 10000")

    # 매장별로 묶어서 revenue 컬럼만 선택
    .groupby("store_id")["revenue"]

    # 매장별 평균 매출 계산 → 결과는 store_id가 인덱스인 Series
    .mean()

    # 소수점 없이 원 단위로 반올림
    .round(0)

    # reset_index(): 인덱스(store_id)를 일반 컬럼으로 내려서 DataFrame으로 변환
    #   name="avg_cake_revenue" → Series 값 컬럼의 이름 지정 (안 쓰면 원래 이름 'revenue'가 됨)
    .reset_index(name="avg_cake_revenue")
)

# 결과 확인: store_id, avg_cake_revenue 두 컬럼의 표
print(cake_kpi)

# Parquet으로 저장
#   index=False → reset_index로 store_id가 이미 컬럼이 됐으니, 0, 1, 2... 행 번호는 저장하지 않음
cake_kpi.to_parquet(OUT_DIR / "cake_kpi.parquet", index=False)

# 저장 경로 출력 (\n → 앞에 빈 줄 하나)
# f-string 안에서 바깥이 큰따옴표라 파일명은 작은따옴표로 감쌈
print(f"\n저장 완료: {OUT_DIR / 'cake_kpi.parquet'}")

  store_id  avg_cake_revenue
0      S01           81722.0
1      S02          134550.0
2      S03           99488.0
3      S04           85946.0
4      S05          104395.0
5      S06           71805.0

저장 완료: outputs/cake_kpi.parquet


In [52]:
import os

# 같은 데이터를 두 형식으로 저장
parquet_path = OUT_DIR / "bakery_clean.parquet"
csv_path = OUT_DIR / "bakery_clean.csv"

bakery_clean.to_parquet(parquet_path, index=False)
bakery_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")

# 파일 크기(바이트) 비교 → KB로 변환해 출력
p_kb = os.path.getsize(parquet_path) / 1024
c_kb = os.path.getsize(csv_path) / 1024
print(f"Parquet: {p_kb:,.1f}KB  /  CSV: {c_kb:,.1f}KB  →  CSV가 약 {c_kb / p_kb:.1f}배 큼")

Parquet: 9.2KB  /  CSV: 83.9KB  →  CSV가 약 9.2배 큼


# 빵셀러 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그
- 기간: 2025-04-01 ~ 2025-06-30 (3개월)
- 원본 행 수 / 정제 후 행 수: (2,164)행 → (1,784)행

## 2. 발견한 품질 문제
- 결측: revenue 컬럼 2.77% (60건)
- 중복: 321건
- 이상치: quantity 50, revenue 음수(-45,000)
- 표기 혼재: store_id 앞뒤 공백(' S01 '), item '케익'/'케이크', date_str 날짜 형식 3종

## 3. 처리 결정과 근거 (5줄로)
1. 중복 행 (321)건 → 완전 중복 기준으로 제거
2. store_id 공백 → strip으로 제거해 같은 매장이 둘로 나뉘지 않게 함
3. item '케익' → '케이크' → 같은 상품이므로 표기 통일
4. revenue 음수 → 별도 보관 (1)건
5. revenue 결측 (58)건 → 제거 (이유: 매출 핵심 값이라 임의로 채우면 합계가 왜곡되고, 비율이 3% 미만으로 작음)

## 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: S06, 2025-04 (9,156,000원)
- (상품 관점) 매출 1위 상품: 쿠키 (26,690,000원)

## 5. 한계와 후속 작업
- quantity 50 이상치가 277건으로 전체 1,784건중 약 16%, 총매출의 약 80%를 차지 → 확인 후 KPI 재산출 필요
- 기간 해석: 월별 수치가 각각 하루치 기록이라 월간 추세로 해석하면 안 됨 → 전체 기간 일자 데이터 확보 필요
- 환불 원인: 환불 데이터의 발생 원인(환불 vs 입력 실수) 분석 필요
- item '케익'/'케이크' 외에 다른 표기 변종이 있는지 수시 점검 필요